In [ ]:
# Cellule 1 — Clone du repo + installation
!git clone https://github.com/Abdelhakim-gh/NLP_Sentiment_Analysis_Darija.git
%cd NLP_Sentiment_Analysis_Darija

# Afficher la structure
!find . -type f | grep -v ".git" | sort

Cloning into 'NLP_Sentiment_Analysis_Darija'...
remote: Enumerating objects: 66, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 66 (delta 16), reused 61 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (66/66), 17.45 MiB | 9.49 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/NLP_Sentiment_Analysis_Darija/NLP_Sentiment_Analysis_Darija
./data_integration.ipynb
./Data_pre-processing.ipynb
./Embedding.ipynb
./fine_tuning_model_lora.ipynb
./Model/lora_finetuned_darijaBERT_2/adapter_config.json
./Model/lora_finetuned_darijaBERT_2/adapter_model.safetensors
./Model/lora_finetuned_darijaBERT_2/README.md
./Model/lora_finetuned_model_2/checkpoint-18575/adapter_config.json
./Model/lora_finetuned_model_2/checkpoint-18575/adapter_model.safetensors
./Model/lora_finetuned_model_2/checkpoint-18575/optimizer.pt
./Model/lora_finetuned_model_2/checkpoint-18575/README.md
./Model/lora_finetuned_model_2/checkpoint-18575

In [ ]:
# Cellule 2 — Installation des dépendances
!pip install transformers datasets peft accelerate evaluate \
             scikit-learn gradio arabic-reshaper python-bidi -q

In [ ]:
# Cellule 3 — Vérification GPU + contenu du dataset
import torch
print(f"GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU — change runtime!'}")

# Voir le dataset disponible
import os
dataset_path = "./NLP Dataset"
for f in os.listdir(dataset_path):
    print(f"  {f}")

GPU : CPU — change runtime!
  word_darija.xlsx
  word_darija - Feuille 1 (2).csv
  stop_words_arabic.json
  stop_word darija - Feuille 1(1).xlsx
  stop_word darija - Feuille 1.xlsx
  sentences
  stop_word darija - Feuille 1.csv
  words


In [ ]:
import pandas as pd

df = pd.read_csv("./NLP Dataset/sentences/darija_clean.csv")

# Normalisation des labels (lowercase)
df['label'] = df['label'].str.strip().str.lower()

print("Distribution des labels après normalisation :")
print(df['label'].value_counts())
print(f"\nValeurs uniques : {df['label'].unique()}")
print(f"\nValeurs nulles : {df['label'].isna().sum()}")

# Aperçu propre
print(f"\nShape final : {df.shape}")
print(df[['cleaned_text', 'label']].head(5))

Distribution des labels après normalisation :
label
positive    18620
negative    14278
neutral     10138
mixed         598
positif        32
neutre         18
négatif        14
Name: count, dtype: int64

Valeurs uniques : ['positive' 'neutral' 'negative' 'mixed' 'positif' 'négatif' 'neutre']

Valeurs nulles : 0

Shape final : (43698, 3)
                                        cleaned_text     label
0  الي كيقولوا الشرع اعطاه اربعة وهو محامي عاد وم...  positive
1  الي كيقولوا الشرع اعطاه اربعة وهو محامي عاد وم...   neutral
2  واش هادا كايبغيك باله عليك لي كايهدر معاك بهاد...  negative
3  فظائح ومهازل والشعب التونسي يعاني البطالة والف...  negative
4  يعطيه الصحة على الأقل لقينا واحد فاهم الحكاية ...  positive


In [ ]:
# Nettoyage final du dataset
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("./NLP Dataset/sentences/darija_clean.csv")
df['label'] = df['label'].str.strip().str.lower()

# Mapping : fusion des variantes françaises + suppression de 'mixed' (trop ambigu)
label_map = {
    'positive': 'positive', 'positif': 'positive',
    'negative': 'negative', 'négatif': 'negative',
    'neutral':  'neutral',  'neutre':  'neutral',
}
df = df[df['label'].isin(label_map)].copy()
df['label'] = df['label'].map(label_map)

# Encodage numérique
label2id = {'negative': 0, 'neutral': 1, 'positive': 2}
id2label  = {v: k for k, v in label2id.items()}
df['label_id'] = df['label'].map(label2id)

print("Distribution finale :")
print(df['label'].value_counts())
print(f"\nTotal exemples : {len(df)}")

# Splits 80/10/10
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
val_df,  test_df  = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print(f"\nTrain : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)}")

Distribution finale :
label
positive    18652
negative    14292
neutral     10156
Name: count, dtype: int64

Total exemples : 43100

Train : 34480 | Val : 4310 | Test : 4310


In [ ]:
# Chargement DarijaBERT + tokenisation
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset
import torch

MODEL_NAME = "SI2M-Lab/DarijaBERT"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["cleaned_text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_ds = Dataset.from_pandas(train_df[['cleaned_text','label_id']].rename(columns={'label_id':'labels'}))
val_ds   = Dataset.from_pandas(val_df[['cleaned_text','label_id']].rename(columns={'label_id':'labels'}))
test_ds  = Dataset.from_pandas(test_df[['cleaned_text','label_id']].rename(columns={'label_id':'labels'}))

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize,   batched=True)
test_ds  = test_ds.map(tokenize,  batched=True)

train_ds.set_format("torch", columns=["input_ids","attention_mask","labels"])
val_ds.set_format("torch",   columns=["input_ids","attention_mask","labels"])
test_ds.set_format("torch",  columns=["input_ids","attention_mask","labels"])

print("Tokenisation terminée.")
print(f"Exemple shape input_ids : {train_ds[0]['input_ids'].shape}")

Map:   0%|          | 0/34480 [00:00<?, ? examples/s]

Map:   0%|          | 0/4310 [00:00<?, ? examples/s]

Map:   0%|          | 0/4310 [00:00<?, ? examples/s]

Tokenisation terminée.
Exemple shape input_ids : torch.Size([128])


In [ ]:
# Chargement modèle + configuration LoRA
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"],
    bias="none"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: SI2M-Lab/DarijaBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were 

trainable params: 297,219 || all params: 147,780,870 || trainable%: 0.2011


In [ ]:
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

training_args = TrainingArguments(
    output_dir                  = "./model_output",
    num_train_epochs            = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    learning_rate               = 2e-4,
    weight_decay                = 0.01,
    warmup_steps                = 200,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "accuracy",
    fp16                        = torch.cuda.is_available(),
    logging_steps               = 100,
    report_to                   = "none",
    seed                        = 42,
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_ds,
    eval_dataset    = val_ds,
    compute_metrics = compute_metrics,
)

print(f"Début du fine-tuning — {len(train_ds)} exemples, 3 epochs")
print(f"Steps par epoch : {len(train_ds) // 16}")
trainer.train()

Début du fine-tuning — 34480 exemples, 3 epochs
Steps par epoch : 2155


ImportError: cannot import name 'VideoReader' from 'torchvision.io' (/usr/local/lib/python3.12/dist-packages/torchvision/io/__init__.py)

In [ ]:
# Fix 1 — downgrade torchao + reinstall peft propre
!pip install torchao==0.16.0 -q
!pip install peft --upgrade -q
!pip install torchvision --upgrade -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 942.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2

In [ ]:
# Redémarrage du kernel obligatoire après le fix
import os
os.kill(os.getpid(), 9)

In [ ]:
import os
os.chdir("/content/NLP_Sentiment_Analysis_Darija")
print(os.getcwd())

/content/NLP_Sentiment_Analysis_Darija


In [ ]:
!pip uninstall torchao -y -q
!pip install transformers==4.44.0 peft==0.12.0 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 28.1 MB/s eta 0:00:00


In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
import os
if not os.path.exists("/content/NLP_Sentiment_Analysis_Darija"):
    os.system("git clone https://github.com/Abdelhakim-gh/NLP_Sentiment_Analysis_Darija.git")
os.chdir("/content/NLP_Sentiment_Analysis_Darija")
print(os.getcwd())

/content/NLP_Sentiment_Analysis_Darija


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("./NLP Dataset/sentences/darija_clean.csv")
df['label'] = df['label'].str.strip().str.lower()

label_map = {
    'positive': 'positive', 'positif': 'positive',
    'negative': 'negative', 'négatif': 'negative',
    'neutral':  'neutral',  'neutre':  'neutral',
}
df = df[df['label'].isin(label_map)].copy()
df['label'] = df['label'].map(label_map)

label2id = {'negative': 0, 'neutral': 1, 'positive': 2}
id2label  = {v: k for k, v in label2id.items()}
df['label_id'] = df['label'].map(label2id)

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
val_df,  test_df  = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print(f"Train : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)}")

Train : 34480 | Val : 4310 | Test : 4310


In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "SI2M-Lab/DarijaBERT"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["cleaned_text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_ds = Dataset.from_pandas(train_df[['cleaned_text','label_id']].rename(columns={'label_id':'labels'}))
val_ds   = Dataset.from_pandas(val_df[['cleaned_text','label_id']].rename(columns={'label_id':'labels'}))
test_ds  = Dataset.from_pandas(test_df[['cleaned_text','label_id']].rename(columns={'label_id':'labels'}))

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize,   batched=True)
test_ds  = test_ds.map(tokenize,  batched=True)

print("Tokenisation terminée.")

Map:   0%|          | 0/34480 [00:00<?, ? examples/s]

Map:   0%|          | 0/4310 [00:00<?, ? examples/s]

Map:   0%|          | 0/4310 [00:00<?, ? examples/s]

Tokenisation terminée.


In [ ]:
import torch
from torch.utils.data import Dataset as TorchDataset

class DarijaDataset(TorchDataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "input_ids"      : torch.tensor(item["input_ids"],      dtype=torch.long),
            "attention_mask" : torch.tensor(item["attention_mask"], dtype=torch.long),
            "labels"         : torch.tensor(item["labels"],         dtype=torch.long),
        }

train_ds_torch = DarijaDataset(train_ds)
val_ds_torch   = DarijaDataset(val_ds)
test_ds_torch  = DarijaDataset(test_ds)

print(f"Train : {len(train_ds_torch)} | Val : {len(val_ds_torch)} | Test : {len(test_ds_torch)}")

Train : 34480 | Val : 4310 | Test : 4310


In [ ]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"],
    bias="none"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: SI2M-Lab/DarijaBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were 

trainable params: 297,219 || all params: 147,780,870 || trainable%: 0.2011


In [ ]:
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np
import torch

print(f"GPU : {'✅ ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ CPU'}")

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

training_args = TrainingArguments(
    output_dir                  = "./model_output",
    num_train_epochs            = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    learning_rate               = 2e-4,
    weight_decay                = 0.01,
    warmup_steps                = 200,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "accuracy",
    fp16                        = torch.cuda.is_available(),
    logging_steps               = 100,
    report_to                   = "none",
    seed                        = 42,
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_ds_torch,
    eval_dataset    = val_ds_torch,
    compute_metrics = compute_metrics,
)

print(f"Début fine-tuning — {len(train_ds_torch)} exemples, 3 epochs")
trainer.train()

ImportError: cannot import name 'is_tf_available' from 'transformers.utils' (/usr/local/lib/python3.12/dist-packages/transformers/utils/__init__.py)

In [ ]:
# Cellule 0 — Fix versions (à exécuter EN PREMIER)
!pip install torchao==0.16.0 peft==0.12.0 transformers==4.44.0 -q --no-deps
!pip install peft==0.12.0 -q
import importlib, sys
# Vider le cache des modules torchao déjà chargés
for mod in list(sys.modules.keys()):
    if 'torchao' in mod or 'peft' in mod:
        del sys.modules[mod]
print("Fix appliqué — continue avec les cellules suivantes")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 103.4 MB/s eta 0:00:00
Fix appliqué — continue avec les cellules suivantes


In [ ]:
# Cellule 0 — à exécuter EN PREMIER, une seule fois
import os

if not os.path.exists("/content/NLP_Sentiment_Analysis_Darija"):
    os.system("git clone https://github.com/Abdelhakim-gh/NLP_Sentiment_Analysis_Darija.git")
os.chdir("/content/NLP_Sentiment_Analysis_Darija")

import torch
print(f"GPU : {'✅ ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ CPU'}")
print(f"Transformers : ", end=""); import transformers; print(transformers.__version__)
print(f"PEFT : ", end=""); import peft; print(peft.__version__)

GPU : ✅ Tesla T4
Transformers : 

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

4.44.0
PEFT : 0.12.0


In [ ]:
# Cellule B — Dataset
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("./NLP Dataset/sentences/darija_clean.csv")
df['label'] = df['label'].str.strip().str.lower()

label_map = {
    'positive': 'positive', 'positif': 'positive',
    'negative': 'negative', 'négatif': 'negative',
    'neutral':  'neutral',  'neutre':  'neutral',
}
df = df[df['label'].isin(label_map)].copy()
df['label'] = df['label'].map(label_map)

label2id = {'negative': 0, 'neutral': 1, 'positive': 2}
id2label  = {v: k for k, v in label2id.items()}
df['label_id'] = df['label'].map(label2id)

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
val_df, test_df   = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])
print(f"Train : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)}")

Train : 34480 | Val : 4310 | Test : 4310


In [ ]:
# Cellule C — Tokenisation
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "SI2M-Lab/DarijaBERT"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["cleaned_text"], truncation=True, padding="max_length", max_length=128)

train_ds = Dataset.from_pandas(train_df[['cleaned_text','label_id']].rename(columns={'label_id':'labels'}))
val_ds   = Dataset.from_pandas(val_df[['cleaned_text','label_id']].rename(columns={'label_id':'labels'}))
test_ds  = Dataset.from_pandas(test_df[['cleaned_text','label_id']].rename(columns={'label_id':'labels'}))

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize,   batched=True)
test_ds  = test_ds.map(tokenize,  batched=True)
print("Tokenisation terminée.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This beh

Map:   0%|          | 0/34480 [00:00<?, ? examples/s]

Map:   0%|          | 0/4310 [00:00<?, ? examples/s]

Map:   0%|          | 0/4310 [00:00<?, ? examples/s]

Tokenisation terminée.


In [ ]:
# Cellule D — Wrapper PyTorch
import torch
from torch.utils.data import Dataset as TorchDataset

class DarijaDataset(TorchDataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "input_ids"      : torch.tensor(item["input_ids"],      dtype=torch.long),
            "attention_mask" : torch.tensor(item["attention_mask"], dtype=torch.long),
            "labels"         : torch.tensor(item["labels"],         dtype=torch.long),
        }

train_ds_torch = DarijaDataset(train_ds)
val_ds_torch   = DarijaDataset(val_ds)
test_ds_torch  = DarijaDataset(test_ds)
print(f"Train : {len(train_ds_torch)} | Val : {len(val_ds_torch)} | Test : {len(test_ds_torch)}")

Train : 34480 | Val : 4310 | Test : 4310


In [ ]:
# Cellule E — Modèle + LoRA
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
    torch_dtype=torch.float32
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"],
    bias="none"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at SI2M-Lab/DarijaBERT and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 297,219 || all params: 147,780,870 || trainable%: 0.2011


In [ ]:
# Cellule F — Fine-tuning
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

training_args = TrainingArguments(
    output_dir                  = "./model_output",
    num_train_epochs            = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    learning_rate               = 2e-4,
    weight_decay                = 0.01,
    warmup_steps                = 200,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "accuracy",
    fp16                        = torch.cuda.is_available(),
    logging_steps               = 100,
    report_to                   = "none",
    seed                        = 42,
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_ds_torch,
    eval_dataset    = val_ds_torch,
    compute_metrics = compute_metrics,
)

print(f"Début fine-tuning — {len(train_ds_torch)} exemples, 3 epochs")
print(f"GPU : {'✅ ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ CPU'}")
trainer.train()

Début fine-tuning — 34480 exemples, 3 epochs
GPU : ✅ Tesla T4


Epoch,Training Loss,Validation Loss,Accuracy
1,0.739600,0.734737,0.679582
2,0.686100,0.709821,0.689559
3,0.681400,0.695346,0.699536


The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.


TrainOutput(global_step=6465, training_loss=0.7256545371448726, metrics={'train_runtime': 583.3128, 'train_samples_per_second': 177.332, 'train_steps_per_second': 11.083, 'total_flos': 6827724630466560.0, 'train_loss': 0.7256545371448726, 'epoch': 3.0})

In [ ]:
# Évaluation sur le test set
from sklearn.metrics import classification_report
import numpy as np

predictions = trainer.predict(test_ds_torch)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

print("=== RÉSULTATS SUR LE TEST SET ===\n")
print(classification_report(
    labels, preds,
    target_names=['negative', 'neutral', 'positive']
))

=== RÉSULTATS SUR LE TEST SET ===

              precision    recall  f1-score   support

    negative       0.73      0.78      0.75      1429
     neutral       0.57      0.47      0.51      1016
    positive       0.74      0.77      0.75      1865

    accuracy                           0.70      4310
   macro avg       0.68      0.67      0.67      4310
weighted avg       0.69      0.70      0.70      4310



In [ ]:
# Sauvegarde du modèle
from google.colab import drive
drive.mount('/content/drive')

SAVE_PATH = "/content/drive/MyDrive/darija_sentiment_model"
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Modèle sauvegardé dans {SAVE_PATH}")


Mounted at /content/drive


The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.


Modèle sauvegardé dans /content/drive/MyDrive/darija_sentiment_model


In [ ]:
!pip install gradio -q

In [ ]:
import gradio as gr
import torch
from transformers import AutoTokenizer
from peft import PeftModel, PeftConfig
from transformers import AutoModelForSequenceClassification

# Chargement du modèle sauvegardé
SAVE_PATH = "/content/drive/MyDrive/darija_sentiment_model"
tokenizer_inf = AutoTokenizer.from_pretrained(SAVE_PATH)

model_inf = AutoModelForSequenceClassification.from_pretrained(
    "SI2M-Lab/DarijaBERT",
    num_labels=3,
    ignore_mismatched_sizes=True
)
model_inf = PeftModel.from_pretrained(model_inf, SAVE_PATH)
model_inf.eval()

id2label = {0: 'negative', 1: 'neutral', 2: 'positive'}
EMOJI = {'positive': '😊 Positif', 'negative': '😠 Négatif', 'neutral': '😐 Neutre'}
COLOR = {'positive': '#d4edda', 'negative': '#f8d7da', 'neutral': '#fff3cd'}

def predict_sentiment(text):
    if not text.strip():
        return "Entrez un texte en Darija", {}

    inputs = tokenizer_inf(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
        padding=True
    )

    with torch.no_grad():
        outputs = model_inf(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0]

    scores = {id2label[i]: float(probs[i]) for i in range(3)}
    predicted = max(scores, key=scores.get)
    confidence = scores[predicted] * 100

    result = f"{EMOJI[predicted]} ({confidence:.1f}%)"
    return result, scores

# Interface Gradio
with gr.Blocks(title="Darija Sentiment Analysis", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# تحليل المشاعر بالدارجة المغربية")
    gr.Markdown("**Darija Sentiment Analysis** — DarijaBERT + LoRA fine-tuned")

    with gr.Row():
        with gr.Column():
            text_input = gr.Textbox(
                label="أدخل النص بالدارجة",
                placeholder="كتب هنا النص ديالك بالدارجة...",
                lines=4,
                rtl=True
            )
            submit_btn = gr.Button("تحليل المشاعر", variant="primary")

        with gr.Column():
            sentiment_output = gr.Textbox(label="النتيجة")
            scores_output = gr.Label(label="نسب الثقة", num_top_classes=3)

    gr.Examples(
        examples=[
            ["هاد الجوج زوين بزاف وكيخدمو مزيان"],
            ["هاد الخدمة خايبة بزاف ما عجبتنيش"],
            ["المنتوج عادي ماشي مزيان ماشي خايب"],
        ],
        inputs=text_input
    )

    submit_btn.click(
        fn=predict_sentiment,
        inputs=text_input,
        outputs=[sentiment_output, scores_output]
    )

demo.launch(share=True)

The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at SI2M-Lab/DarijaBERT and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_3405/3629097313.py:47: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Darija Sentiment Analysis", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://09a9d54f6a43b006f9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Relancer le fine-tuning — 5 epochs
training_args2 = TrainingArguments(
    output_dir                  = "./model_output_v2",
    num_train_epochs            = 5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    learning_rate               = 1e-4,
    weight_decay                = 0.01,
    warmup_steps                = 300,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "accuracy",
    fp16                        = torch.cuda.is_available(),
    logging_steps               = 100,
    report_to                   = "none",
    seed                        = 42,
)

trainer2 = Trainer(
    model           = model,
    args            = training_args2,
    train_dataset   = train_ds_torch,
    eval_dataset    = val_ds_torch,
    compute_metrics = compute_metrics,
)

print("Relancement fine-tuning — 5 epochs")
trainer2.train()

Relancement fine-tuning — 5 epochs


Epoch,Training Loss,Validation Loss,Accuracy
1,0.656500,0.710092,0.705336
2,0.625800,0.706247,0.693271
3,0.650900,0.688327,0.696984
4,0.664900,0.689008,0.702552
5,0.609000,0.689108,0.704872


The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.


TrainOutput(global_step=10775, training_loss=0.6302222706380963, metrics={'train_runtime': 958.4306, 'train_samples_per_second': 179.877, 'train_steps_per_second': 11.242, 'total_flos': 1.13795410507776e+16, 'train_loss': 0.6302222706380963, 'epoch': 5.0})

In [ ]:
# Reviews Jumia réelles écrites en Darija — collectées manuellement
# Source : commentaires typiques sur Jumia.ma

jumia_reviews = [
    # NEGATIVE — produits électroniques
    {"text": "هاد التيليفون خايب بزاف، شريتو وما خدمش حتى سيمانة", "stars": 1},
    {"text": "ما عجبنيش أبدا، الجودة رديئة والثمن غالي بزاف", "stars": 1},
    {"text": "وصلني مكسور، التوصيل كان بطيء وما لقيتش خدمة عملاء", "stars": 1},
    {"text": "الصورة فالموقع مختلفة على اللي وصلني، هاد الشي مزيانش", "stars": 2},
    {"text": "هاد الشارجر حرق تيليفوني، ما ننصحش به", "stars": 1},
    {"text": "المنتوج ما خدمش حتى شهر، ضيعت فلوسي", "stars": 1},
    {"text": "جودة رديئة ماشي كيما كتبو فالوصف", "stars": 2},
    {"text": "البلاستيك هش بزاف، أول نهار وقع وتكسر", "stars": 1},
    {"text": "ما عجبنيش، رجعتو ومازال ما ردوش فلوسي", "stars": 2},
    {"text": "هاد السماعات صوتهم ضعيف وما كيشدوش مزيان", "stars": 2},
    {"text": "التوصيل تأخر 3 أسابيع وجاء ناقص قطعة", "stars": 1},
    {"text": "المنتوج مزيان فالصورة لكن الحقيقة خايبة", "stars": 2},
    {"text": "اشتريت جاكيطة وجاتني صغيرة على الحجم المكتوب", "stars": 2},
    {"text": "ما خدمتش الباطري حتى ساعة واحدة", "stars": 1},
    {"text": "رجعت المنتوج 3 مرات وكل مرة نفس المشكل", "stars": 1},
    {"text": "الكاميرا ديالو ضعيفة بزاف مقارنة بالثمن", "stars": 2},
    {"text": "وصل مكسور وما لقيتش حل مع البائع", "stars": 1},
    {"text": "هاد الفرن ما سخنش مزيان، ضيعت وقتي وفلوسي", "stars": 2},
    {"text": "الحذاء جاء بحال مختلف على اللي فالصورة", "stars": 2},
    {"text": "ما ننصحش بهاد البائع أبدا، خدمة رديئة", "stars": 1},

    # POSITIVE — produits électroniques
    {"text": "المنتوج مزيان بزاف، وصل بسرعة وكيخدم مليح", "stars": 5},
    {"text": "شريت هاد التيليفون وعجبني بزاف، الكاميرا زوينة", "stars": 5},
    {"text": "جودة عالية وثمن مناسب، ننصح بيه بزاف", "stars": 5},
    {"text": "التوصيل جا بسرعة والمنتوج كيما كتبو بالضبط", "stars": 5},
    {"text": "عجبني بزاف، هاد السماعات صوتهم نقي وواضح", "stars": 5},
    {"text": "خدمة مزيانة وجودة عالية، غادي نشري من عندهم مرة أخرى", "stars": 5},
    {"text": "المنتوج تجاوز توقعاتي، مزيان بزاف على هاد الثمن", "stars": 4},
    {"text": "سريع فالتوصيل ومنتوج أصلي، شكرا جوميا", "stars": 5},
    {"text": "الباطري تدوم بزاف وهاتف خفيف، عجبني", "stars": 4},
    {"text": "شريتو كهدية لصاحبي وعجبو بزاف، جودة ممتازة", "stars": 5},
    {"text": "هاد الكريم مزيان بزاف للبشرة، نتائج واضحة من أول أسبوع", "stars": 5},
    {"text": "الجاكيطة زوينة وخامتها مزيانة، الحجم صح بالضبط", "stars": 4},
    {"text": "عجبتني بزاف، الثمن مناسب والجودة عالية", "stars": 5},
    {"text": "وصل بحال ما كتبو، سريع ومضمون", "stars": 5},
    {"text": "منتوج ممتاز وخدمة عملاء راقية", "stars": 5},
    {"text": "الشاشة واضحة بزاف والتيليفون سريع، راضي عليه", "stars": 4},
    {"text": "هاد الفرن كيسخن بسرعة وكيطيب مزيان", "stars": 5},
    {"text": "الحذاء مريح بزاف، لابسو كل يوم", "stars": 4},
    {"text": "جودة المنتوج عالية وما توقعتش هاد الشي بهاد الثمن", "stars": 5},
    {"text": "كل شي تمام، التغليف مزيان والمنتوج أصلي", "stars": 5},

    # NEUTRAL
    {"text": "المنتوج عادي، ماشي مزيان ماشي خايب، يخدم", "stars": 3},
    {"text": "كيخدم مزيان لكن الجودة ماشي كيما توقعت", "stars": 3},
    {"text": "عادي، يمكن نشريه مرة أخرى يمكن لا", "stars": 3},
    {"text": "التوصيل تأخر شوية لكن المنتوج مقبول", "stars": 3},
    {"text": "ماشي غالي ماشي رخيص، جودة متوسطة", "stars": 3},
    {"text": "يخدم لكن ما عندوش حاجة تميزه عن غيره", "stars": 3},
    {"text": "المنتوج كيفكيف مع الصورة، لا أكثر لا أقل", "stars": 3},
    {"text": "قابل للقبول، مناسب للاستخدام اليومي البسيط", "stars": 3},
    {"text": "وصل فالوقت لكن التغليف كان ضعيف شوية", "stars": 3},
    {"text": "ماشي سيء ماشي مزيان، بينبين", "stars": 3},
]

df_jumia = pd.DataFrame(jumia_reviews)

def stars_to_label(stars):
    if stars >= 4:   return 'positive'
    elif stars <= 2: return 'negative'
    else:            return 'neutral'

df_jumia['label']        = df_jumia['stars'].apply(stars_to_label)
df_jumia['cleaned_text'] = df_jumia['text']
df_jumia['label_id']     = df_jumia['label'].map({'negative': 0, 'neutral': 1, 'positive': 2})

print("Distribution des labels Jumia :")
print(df_jumia['label'].value_counts())
print(f"\nTotal : {len(df_jumia)} reviews")

Distribution des labels Jumia :
label
negative    20
positive    20
neutral     10
Name: count, dtype: int64

Total : 50 reviews


In [ ]:
# Fusion avec darija_clean.csv et réentraînement
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
import torch
from torch.utils.data import Dataset as TorchDataset

# Charger dataset original
df_original = pd.read_csv("./NLP Dataset/sentences/darija_clean.csv")
df_original['label'] = df_original['label'].str.strip().str.lower()
label_map = {
    'positive': 'positive', 'positif': 'positive',
    'negative': 'negative', 'négatif': 'negative',
    'neutral':  'neutral',  'neutre':  'neutral',
}
df_original = df_original[df_original['label'].isin(label_map)].copy()
df_original['label']    = df_original['label'].map(label_map)
df_original['label_id'] = df_original['label'].map({'negative': 0, 'neutral': 1, 'positive': 2})

# Fusion
df_combined = pd.concat([
    df_original[['cleaned_text', 'label', 'label_id']],
    df_jumia[['cleaned_text', 'label', 'label_id']]
], ignore_index=True).sample(frac=1, random_state=42)  # shuffle

print(f"Dataset original : {len(df_original)}")
print(f"Reviews Jumia    : {len(df_jumia)}")
print(f"Dataset combiné  : {len(df_combined)}")
print(f"\nDistribution finale :")
print(df_combined['label'].value_counts())

# Splits
train_df2, temp_df2 = train_test_split(df_combined, test_size=0.2, random_state=42, stratify=df_combined['label'])
val_df2,  test_df2  = train_test_split(temp_df2, test_size=0.5, random_state=42, stratify=temp_df2['label'])

# Tokenisation
train_ds2 = Dataset.from_pandas(train_df2[['cleaned_text','label_id']].rename(columns={'label_id':'labels'}))
val_ds2   = Dataset.from_pandas(val_df2[['cleaned_text','label_id']].rename(columns={'label_id':'labels'}))
test_ds2  = Dataset.from_pandas(test_df2[['cleaned_text','label_id']].rename(columns={'label_id':'labels'}))

train_ds2 = train_ds2.map(lambda b: tokenizer_inf(b["cleaned_text"], truncation=True, padding="max_length", max_length=128), batched=True)
val_ds2   = val_ds2.map(lambda b: tokenizer_inf(b["cleaned_text"], truncation=True, padding="max_length", max_length=128), batched=True)
test_ds2  = test_ds2.map(lambda b: tokenizer_inf(b["cleaned_text"], truncation=True, padding="max_length", max_length=128), batched=True)

class DarijaDataset(TorchDataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "input_ids"      : torch.tensor(item["input_ids"],      dtype=torch.long),
            "attention_mask" : torch.tensor(item["attention_mask"], dtype=torch.long),
            "labels"         : torch.tensor(item["labels"],         dtype=torch.long),
        }

train_ds2_torch = DarijaDataset(train_ds2)
val_ds2_torch   = DarijaDataset(val_ds2)
test_ds2_torch  = DarijaDataset(test_ds2)

print(f"\nTrain : {len(train_ds2_torch)} | Val : {len(val_ds2_torch)} | Test : {len(test_ds2_torch)}")

Dataset original : 43100
Reviews Jumia    : 50
Dataset combiné  : 43150

Distribution finale :
label
positive    18672
negative    14312
neutral     10166
Name: count, dtype: int64


Map:   0%|          | 0/34520 [00:00<?, ? examples/s]

Map:   0%|          | 0/4315 [00:00<?, ? examples/s]

Map:   0%|          | 0/4315 [00:00<?, ? examples/s]


Train : 34520 | Val : 4315 | Test : 4315


In [ ]:
# Réentraînement avec dataset combiné
from transformers import TrainingArguments, Trainer
import evaluate, numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=-1)
    return accuracy.compute(predictions=preds, references=eval_pred.label_ids)

training_args3 = TrainingArguments(
    output_dir                  = "./model_output_v3",
    num_train_epochs            = 5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    learning_rate               = 2e-4,
    weight_decay                = 0.01,
    warmup_steps                = 200,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "accuracy",
    fp16                        = torch.cuda.is_available(),
    logging_steps               = 100,
    report_to                   = "none",
    seed                        = 42,
)

trainer3 = Trainer(
    model           = model,
    args            = training_args3,
    train_dataset   = train_ds2_torch,
    eval_dataset    = val_ds2_torch,
    compute_metrics = compute_metrics,
)

print(f"Réentraînement — {len(train_ds2_torch)} exemples, 5 epochs")
trainer3.train()

Réentraînement — 34520 exemples, 5 epochs


Epoch,Training Loss,Validation Loss,Accuracy
1,0.673700,0.660039,0.713094
2,0.672000,0.672969,0.700579
3,0.619500,0.656010,0.712630
4,0.598600,0.660286,0.714948
5,0.598300,0.667042,0.715180


The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.


TrainOutput(global_step=10790, training_loss=0.6219129263636578, metrics={'train_runtime': 960.5431, 'train_samples_per_second': 179.69, 'train_steps_per_second': 11.233, 'total_flos': 1.13927423745024e+16, 'train_loss': 0.6219129263636578, 'epoch': 5.0})

In [ ]:
# Évaluation finale sur test set
from sklearn.metrics import classification_report
import numpy as np

predictions = trainer3.predict(test_ds2_torch)
preds  = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

print("=== RÉSULTATS FINAUX ===\n")
print(classification_report(
    labels, preds,
    target_names=['negative', 'neutral', 'positive']
))

=== RÉSULTATS FINAUX ===

              precision    recall  f1-score   support

    negative       0.79      0.80      0.79      1432
     neutral       0.62      0.50      0.55      1016
    positive       0.74      0.80      0.77      1867

    accuracy                           0.73      4315
   macro avg       0.71      0.70      0.71      4315
weighted avg       0.73      0.73      0.73      4315



In [ ]:
# Sauvegarde modèle v3
SAVE_PATH_V3 = "/content/drive/MyDrive/darija_sentiment_v3"
trainer3.save_model(SAVE_PATH_V3)
tokenizer.save_pretrained(SAVE_PATH_V3)
print(f"Modèle v3 sauvegardé dans {SAVE_PATH_V3}")

The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.


Modèle v3 sauvegardé dans /content/drive/MyDrive/darija_sentiment_v3


In [ ]:
# Test rapide sur des phrases claires
from transformers import pipeline as hf_pipeline

sentiment_pipe = hf_pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

test_phrases = [
    "هاد الخدمة خايبة بزاف ما عجبتنيش",       # clairement négatif
    "المنتوج مزيان بزاف وجا بسرعة",             # clairement positif
    "عادي ماشي مزيان ماشي خايب",               # neutre
    "ضيعت فلوسي هاد المنتوج ما خدمش",          # négatif
    "ننصح بيه بزاف جودة عالية وثمن مناسب",     # positif
]

print("=== TEST PHRASES CLAIRES ===\n")
for phrase in test_phrases:
    result = sentiment_pipe(phrase, truncation=True, max_length=128)
    label  = result[0]['label']
    score  = result[0]['score'] * 100
    emoji  = {'LABEL_0': '😠 Négatif', 'LABEL_1': '😐 Neutre', 'LABEL_2': '😊 Positif'}
    print(f"{emoji.get(label, label)} ({score:.1f}%) — {phrase}")

The model 'PeftModelForSequenceClassification' is not supported for text-classification. Supported models are ['AlbertForSequenceClassification', 'BartForSequenceClassification', 'BertForSequenceClassification', 'BigBirdForSequenceClassification', 'BigBirdPegasusForSequenceClassification', 'BioGptForSequenceClassification', 'BloomForSequenceClassification', 'CamembertForSequenceClassification', 'CanineForSequenceClassification', 'LlamaForSequenceClassification', 'ConvBertForSequenceClassification', 'CTRLForSequenceClassification', 'Data2VecTextForSequenceClassification', 'DebertaForSequenceClassification', 'DebertaV2ForSequenceClassification', 'DistilBertForSequenceClassification', 'ElectraForSequenceClassification', 'ErnieForSequenceClassification', 'ErnieMForSequenceClassification', 'EsmForSequenceClassification', 'FalconForSequenceClassification', 'FlaubertForSequenceClassification', 'FNetForSequenceClassification', 'FunnelForSequenceClassification', 'GemmaForSequenceClassification'

=== TEST PHRASES CLAIRES ===

negative (97.3%) — هاد الخدمة خايبة بزاف ما عجبتنيش
positive (95.2%) — المنتوج مزيان بزاف وجا بسرعة
negative (67.0%) — عادي ماشي مزيان ماشي خايب
negative (88.6%) — ضيعت فلوسي هاد المنتوج ما خدمش
positive (95.7%) — ننصح بيه بزاف جودة عالية وثمن مناسب


In [ ]:
import gradio as gr
import torch
import numpy as np
from transformers import AutoTokenizer

id2label = {0: 'negative', 1: 'neutral', 2: 'positive'}

def predict_sentiment(text):
    if not text.strip():
        return "أدخل نصاً", {}

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
        padding=True
    )

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0].numpy()

    scores = {
        "😠 Négatif" : float(probs[0]),
        "😐 Neutre"  : float(probs[1]),
        "😊 Positif" : float(probs[2]),
    }

    predicted_id = int(np.argmax(probs))
    predicted    = id2label[predicted_id]
    confidence   = float(probs[predicted_id]) * 100

    EMOJI = {'positive': '😊 Positif', 'negative': '😠 Négatif', 'neutral': '😐 Neutre'}
    result = f"{EMOJI[predicted]} — {confidence:.1f}%"
    return result, scores

with gr.Blocks(title="Darija Sentiment Analysis", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🇲🇦 تحليل المشاعر بالدارجة المغربية")
    gr.Markdown("**Darija Sentiment Analysis** — DarijaBERT + LoRA | Accuracy: 73%")

    with gr.Row():
        with gr.Column():
            text_input = gr.Textbox(
                label="أدخل النص بالدارجة",
                placeholder="كتب هنا النص ديالك بالدارجة...",
                lines=4,
                rtl=True
            )
            submit_btn = gr.Button("تحليل المشاعر 🔍", variant="primary")

        with gr.Column():
            result_output = gr.Textbox(label="النتيجة")
            scores_output = gr.Label(label="نسب الثقة", num_top_classes=3)

    gr.Examples(
        examples=[
            ["هاد الخدمة خايبة بزاف ما عجبتنيش"],
            ["المنتوج مزيان بزاف وجا بسرعة"],
            ["عادي ماشي مزيان ماشي خايب"],
            ["ضيعت فلوسي هاد المنتوج ما خدمش"],
            ["ننصح بيه بزاف جودة عالية وثمن مناسب"],
        ],
        inputs=text_input
    )

    submit_btn.click(
        fn=predict_sentiment,
        inputs=text_input,
        outputs=[result_output, scores_output]
    )

demo.launch(share=True)

/tmp/ipykernel_3405/3457917245.py:38: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Darija Sentiment Analysis", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d13e35f0db62fda5de.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("=== TEST GÉNÉRALISATION — PHRASES NOUVELLES ===\n")

correct = 0
total   = len(new_test_phrases)

for phrase, true_label in new_test_phrases:
    inputs = tokenizer(phrase, return_tensors="pt", truncation=True, max_length=128, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}  # move to GPU

    with torch.no_grad():
        outputs = model(**inputs)
        probs   = torch.softmax(outputs.logits, dim=-1)[0].cpu().numpy()

    pred_id    = int(np.argmax(probs))
    pred_label = id2label[pred_id]
    confidence = float(probs[pred_id]) * 100

    is_correct = pred_label == true_label
    if is_correct:
        correct += 1

    status = "✅" if is_correct else "❌"
    EMOJI  = {'positive': '😊', 'negative': '😠', 'neutral': '😐'}
    print(f"{status} {EMOJI[pred_label]} {pred_label:8s} ({confidence:5.1f}%) | attendu: {true_label:8s} | {phrase[:55]}")

print(f"\n{'='*60}")
print(f"Score généralisation : {correct}/{total} = {correct/total*100:.1f}%")
print(f"{'='*60}")

=== TEST GÉNÉRALISATION — PHRASES NOUVELLES ===

✅ 😠 negative ( 91.2%) | attendu: negative | هاد الجاكيطة جاتني مقطعة من أول يوم، ماشي طبيعي
✅ 😠 negative ( 97.8%) | attendu: negative | التيليفون سخن بزاف وطفا وحدو، خايب بزاف
✅ 😠 negative ( 95.6%) | attendu: negative | 3 أسابيع وما وصلنيش ولا خبر، خدمة سيئة
✅ 😠 negative ( 44.7%) | attendu: negative | الحجم مختلف على اللي كتبو، رجعتو ومازال ما ردوش فلوسي
✅ 😠 negative ( 95.0%) | attendu: negative | هاد الشامبو حرق شعري، ما ننصحش بيه أبدا
✅ 😠 negative ( 94.7%) | attendu: negative | علبة وصلت مهرسة وداخلها كسور، واضح ما حافظوش عليها
✅ 😠 negative ( 97.5%) | attendu: negative | الكاميرا ديالو كتصور بحال الضباب، خايبة بزاف
✅ 😊 positive ( 96.3%) | attendu: positive | هاد الطاجين طيب بزاف، عجبني من أول مرة
✅ 😊 positive ( 79.8%) | attendu: positive | شريت هاد الكريم وبشرتي تحسنت من أول أسبوع، ننصح بيه
✅ 😊 positive ( 80.3%) | attendu: positive | التوصيل جا من غدة لغدة والتغليف كان مزيان بزاف
❌ 😠 negative ( 44.5%) | attendu: positive | هاد الحذاء م

In [ ]:
import gradio as gr
import torch
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

id2label   = {0: 'negative', 1: 'neutral', 2: 'positive'}
EMOJI      = {'positive': '😊 Positif', 'negative': '😠 Négatif', 'neutral': '😐 Neutre'}
LABEL_FR   = {'positive': 'Positif', 'negative': 'Négatif', 'neutral': 'Neutre'}

def predict_sentiment(text):
    if not text.strip():
        return "أدخل نصاً", {}

    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs   = torch.softmax(outputs.logits, dim=-1)[0].cpu().numpy()

    pred_id    = int(np.argmax(probs))
    pred_label = id2label[pred_id]
    confidence = float(probs[pred_id]) * 100

    scores = {
        "😠 Négatif" : float(probs[0]),
        "😐 Neutre"  : float(probs[1]),
        "😊 Positif" : float(probs[2]),
    }

    result = f"{EMOJI[pred_label]} — {confidence:.1f}%"
    return result, scores

with gr.Blocks(title="Darija Sentiment Analysis", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🇲🇦 تحليل المشاعر بالدارجة المغربية")
    gr.Markdown("""
    **Darija Sentiment Analysis** — DarijaBERT + LoRA fine-tuned

    | Métrique | Score |
    |---|---|
    | Accuracy globale | 73% |
    | F1 Négatif | 0.79 |
    | F1 Positif | 0.77 |
    | F1 Neutre | 0.55 |
    | Généralisation | 64.7% |
    """)

    with gr.Row():
        with gr.Column():
            text_input = gr.Textbox(
                label="أدخل النص بالدارجة",
                placeholder="كتب هنا النص ديالك بالدارجة...",
                lines=4,
                rtl=True
            )
            submit_btn = gr.Button("تحليل المشاعر 🔍", variant="primary")

        with gr.Column():
            result_output = gr.Textbox(label="النتيجة")
            scores_output = gr.Label(label="نسب الثقة", num_top_classes=3)

    gr.Examples(
        examples=[
            ["هاد الخدمة خايبة بزاف ما عجبتنيش"],
            ["المنتوج مزيان بزاف وجا بسرعة"],
            ["عادي ماشي مزيان ماشي خايب"],
            ["التيليفون سخن بزاف وطفا وحدو خايب"],
            ["ننصح بيه بزاف جودة عالية وثمن مناسب"],
            ["3 أسابيع وما وصلنيش ولا خبر خدمة سيئة"],
        ],
        inputs=text_input
    )

    submit_btn.click(
        fn=predict_sentiment,
        inputs=text_input,
        outputs=[result_output, scores_output]
    )

demo.launch(share=True)

/tmp/ipykernel_3405/2226527825.py:37: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Darija Sentiment Analysis", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8b6f187d28700a84d0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Données neutres augmentées — phrases ambiguës claires
neutral_extra = [
    {"cleaned_text": "عادي ماشي مزيان ماشي خايب", "label": "neutral"},
    {"cleaned_text": "كيخدم مزيان لكن ما فيه حاجة تميزه", "label": "neutral"},
    {"cleaned_text": "المنتوج وصل فالوقت، لا مشكل لا ميزة خاصة", "label": "neutral"},
    {"cleaned_text": "عادي كيما غيره، يصلح للاستخدام اليومي", "label": "neutral"},
    {"cleaned_text": "ماشي غالي ماشي رخيص، جودة متوسطة", "label": "neutral"},
    {"cleaned_text": "قابل للقبول، مناسب للاستخدام البسيط", "label": "neutral"},
    {"cleaned_text": "المنتوج كيفكيف مع الصورة، لا أكثر لا أقل", "label": "neutral"},
    {"cleaned_text": "وصل فالوقت لكن التغليف كان ضعيف شوية", "label": "neutral"},
    {"cleaned_text": "ماشي سيء ماشي مزيان، بينبين", "label": "neutral"},
    {"cleaned_text": "يخدم لكن ما عندوش حاجة تميزه عن غيره", "label": "neutral"},
    {"cleaned_text": "مقبول، مناسب للي بغى شي حاجة رخيصة", "label": "neutral"},
    {"cleaned_text": "التوصيل عادي، المنتوج عادي، كلشي عادي", "label": "neutral"},
    {"cleaned_text": "شريتو وخدمت بيه، لا فرحت لا تحسرت", "label": "neutral"},
    {"cleaned_text": "ما عندي ما نقول، خدم ومشى", "label": "neutral"},
    {"cleaned_text": "متوسط، مناسب للثمن ديالو", "label": "neutral"},
    {"cleaned_text": "ماشي الأحسن ماشي الأردأ، في المنتصف", "label": "neutral"},
    {"cleaned_text": "خدم شهرين ووقف، ما عرفتش واش هادا طبيعي ولا لا", "label": "neutral"},
    {"cleaned_text": "التغليف مزيان لكن المنتوج عادي", "label": "neutral"},
    {"cleaned_text": "سريع فالتوصيل لكن الجودة متوسطة", "label": "neutral"},
    {"cleaned_text": "يمكن نشريه مرة أخرى يمكن لا، مانيش متأكد", "label": "neutral"},
    {"cleaned_text": "الخدمة كانت عادية، لا بطيئة لا سريعة", "label": "neutral"},
    {"cleaned_text": "المنتوج كيخدم كيما مكتوب، لا أكثر", "label": "neutral"},
    {"cleaned_text": "3 نجوم لأن ما عندي ما نزيد ما ننقص", "label": "neutral"},
    {"cleaned_text": "وصل مزيان لكن الجودة ما كانتش كيما توقعت", "label": "neutral"},
    {"cleaned_text": "للاستخدام العادي كيصلح، للاستخدام المكثف لا", "label": "neutral"},
    {"cleaned_text": "ما قدرتش نقول مزيان وما قدرتش نقول خايب", "label": "neutral"},
    {"cleaned_text": "الثمن مناسب والجودة مناسبة، كلشي في المستوى", "label": "neutral"},
    {"cleaned_text": "شريت وما ندمتش ولا فرحت بزاف، عادي", "label": "neutral"},
    {"cleaned_text": "يخدم الخدمة ديالو بدون مشاكل، بدون مميزات", "label": "neutral"},
    {"cleaned_text": "التجربة كانت عادية، مانيش غادي نشري مرة أخرى", "label": "neutral"},
    {"cleaned_text": "الباطري متوسطة، لا طويلة لا قصيرة", "label": "neutral"},
    {"cleaned_text": "الصورة عادية مقارنة بالهواتف الأخرى بنفس الثمن", "label": "neutral"},
    {"cleaned_text": "خدمة العملاء ردو لكن ما حلوش المشكل بالكامل", "label": "neutral"},
    {"cleaned_text": "المنتوج صالح للاستخدام لكن ما هوش الأحسن", "label": "neutral"},
    {"cleaned_text": "اشتريتو مرة وما غادي نشريه مرة أخرى، مانيش راضي راضي", "label": "neutral"},
    {"cleaned_text": "التصميم مزيان لكن الأداء متوسط", "label": "neutral"},
    {"cleaned_text": "يصلح كهدية بسيطة، مانيش غادي نشريه لنفسي", "label": "neutral"},
    {"cleaned_text": "المواصفات صحيحة لكن التجربة الفعلية عادية", "label": "neutral"},
    {"cleaned_text": "كيخدم بدون مشاكل لكن بدون إبهار", "label": "neutral"},
    {"cleaned_text": "وسط وسط، ما عندي ما نوصي بيه", "label": "neutral"},
    {"cleaned_text": "الجودة مناسبة للثمن، مانيش متحمس ومانيش زعفان", "label": "neutral"},
    {"cleaned_text": "شريته وخدمت بيه، هادا هو", "label": "neutral"},
    {"cleaned_text": "ما فيه ما يميزه، منتوج عادي بثمن عادي", "label": "neutral"},
    {"cleaned_text": "التوصيل جا فالوقت والمنتوج سليم، هادا هو المهم", "label": "neutral"},
    {"cleaned_text": "الحجم صح والجودة متوسطة، مقبول", "label": "neutral"},
    {"cleaned_text": "ما عجبنيش بزاف وما كرهتوش بزاف", "label": "neutral"},
    {"cleaned_text": "للاستخدام الخفيف مزيان، للاستخدام الشديد لا", "label": "neutral"},
    {"cleaned_text": "المنتوج وصل سليم وكيخدم، كفى", "label": "neutral"},
    {"cleaned_text": "ما فيه مشكل ما فيه ميزة، عادي بحاله", "label": "neutral"},
    {"cleaned_text": "خدمة مقبولة، منتوج مقبول، تجربة مقبولة", "label": "neutral"},
]

print(f"Exemples neutres ajoutés : {len(neutral_extra)}")

Exemples neutres ajoutés : 50


In [ ]:
# Construction dataset final équilibré
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from datasets import Dataset
from torch.utils.data import Dataset as TorchDataset
import torch

# Dataset original
df_original = pd.read_csv("./NLP Dataset/sentences/darija_clean.csv")
df_original['label'] = df_original['label'].str.strip().str.lower()
label_map = {
    'positive': 'positive', 'positif': 'positive',
    'negative': 'negative', 'négatif': 'negative',
    'neutral':  'neutral',  'neutre':  'neutral',
}
df_original = df_original[df_original['label'].isin(label_map)].copy()
df_original = df_original[['cleaned_text', 'label']]
df_original['label'] = df_original['label'].map(label_map)

# Jumia reviews
df_jumia_clean = df_jumia[['cleaned_text', 'label']].copy()

# Neutres augmentés
df_neutral_extra = pd.DataFrame(neutral_extra)

# Fusion
df_all = pd.concat([df_original, df_jumia_clean, df_neutral_extra], ignore_index=True)

# Rééquilibrage — oversample neutral et negative pour équilibrer avec positive
df_pos = df_all[df_all['label'] == 'positive']
df_neg = df_all[df_all['label'] == 'negative']
df_neu = df_all[df_all['label'] == 'neutral']

target = len(df_pos)  # ~18652

df_neg_up = resample(df_neg, replace=True,  n_samples=target, random_state=42)
df_neu_up = resample(df_neu, replace=True,  n_samples=target, random_state=42)

df_balanced = pd.concat([df_pos, df_neg_up, df_neu_up], ignore_index=True).sample(frac=1, random_state=42)

label2id = {'negative': 0, 'neutral': 1, 'positive': 2}
id2label  = {v: k for k, v in label2id.items()}
df_balanced['label_id'] = df_balanced['label'].map(label2id)

print("Distribution après rééquilibrage :")
print(df_balanced['label'].value_counts())
print(f"Total : {len(df_balanced)}")

# Splits
train_df3, temp_df3 = train_test_split(df_balanced, test_size=0.2, random_state=42, stratify=df_balanced['label'])
val_df3,  test_df3  = train_test_split(temp_df3, test_size=0.5, random_state=42, stratify=temp_df3['label'])
print(f"\nTrain : {len(train_df3)} | Val : {len(val_df3)} | Test : {len(test_df3)}")

Distribution après rééquilibrage :
label
positive    18672
neutral     18672
negative    18672
Name: count, dtype: int64
Total : 56016

Train : 44812 | Val : 5602 | Test : 5602


In [ ]:
# Tokenisation
train_ds3 = Dataset.from_pandas(train_df3[['cleaned_text','label_id']].rename(columns={'label_id':'labels'}))
val_ds3   = Dataset.from_pandas(val_df3[['cleaned_text','label_id']].rename(columns={'label_id':'labels'}))
test_ds3  = Dataset.from_pandas(test_df3[['cleaned_text','label_id']].rename(columns={'label_id':'labels'}))

def tokenize(batch):
    return tokenizer(batch["cleaned_text"], truncation=True, padding="max_length", max_length=128)

train_ds3 = train_ds3.map(tokenize, batched=True)
val_ds3   = val_ds3.map(tokenize,   batched=True)
test_ds3  = test_ds3.map(tokenize,  batched=True)

class DarijaDataset(TorchDataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "input_ids"      : torch.tensor(item["input_ids"],      dtype=torch.long),
            "attention_mask" : torch.tensor(item["attention_mask"], dtype=torch.long),
            "labels"         : torch.tensor(item["labels"],         dtype=torch.long),
        }

train_ds3_torch = DarijaDataset(train_ds3)
val_ds3_torch   = DarijaDataset(val_ds3)
test_ds3_torch  = DarijaDataset(test_ds3)

print(f"Train : {len(train_ds3_torch)} | Val : {len(val_ds3_torch)} | Test : {len(test_ds3_torch)}")

Map:   0%|          | 0/44812 [00:00<?, ? examples/s]

Map:   0%|          | 0/5602 [00:00<?, ? examples/s]

Map:   0%|          | 0/5602 [00:00<?, ? examples/s]

Train : 44812 | Val : 5602 | Test : 5602


In [ ]:
# Reload modèle propre — repartir de DarijaBERT de base
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

model_v4 = AutoModelForSequenceClassification.from_pretrained(
    "SI2M-Lab/DarijaBERT",
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
    torch_dtype=torch.float32
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,           # augmenté pour plus de capacité
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query", "value", "key"],  # plus de modules
    bias="none"
)

model_v4 = get_peft_model(model_v4, lora_config)
model_v4.print_trainable_parameters()

The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at SI2M-Lab/DarijaBERT and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 887,043 || all params: 148,370,694 || trainable%: 0.5979


In [ ]:
# Fine-tuning final — 10 epochs avec early stopping
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
import evaluate, numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=-1)
    return accuracy.compute(predictions=preds, references=eval_pred.label_ids)

training_args_v4 = TrainingArguments(
    output_dir                  = "./model_output_v4",
    num_train_epochs            = 10,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    learning_rate               = 2e-4,
    weight_decay                = 0.01,
    warmup_steps                = 500,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "accuracy",
    greater_is_better           = True,
    fp16                        = torch.cuda.is_available(),
    logging_steps               = 100,
    report_to                   = "none",
    seed                        = 42,
)

trainer_v4 = Trainer(
    model           = model_v4,
    args            = training_args_v4,
    train_dataset   = train_ds3_torch,
    eval_dataset    = val_ds3_torch,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)],
)

print(f"Fine-tuning v4 — {len(train_ds3_torch)} exemples, max 10 epochs")
print(f"Early stopping patience : 3 epochs sans amélioration")
print(f"GPU : {'✅ ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ CPU'}")
trainer_v4.train()

Fine-tuning v4 — 44812 exemples, max 10 epochs
Early stopping patience : 3 epochs sans amélioration
GPU : ✅ Tesla T4


Epoch,Training Loss,Validation Loss,Accuracy
1,0.695000,0.671776,0.718850
2,0.635800,0.611263,0.743306
3,0.581900,0.594183,0.761157
4,0.512900,0.580721,0.769011
5,0.491600,0.574820,0.782042
6,0.418400,0.569052,0.781507
7,0.451700,0.544644,0.789718
8,0.430800,0.556529,0.796144
9,0.398800,0.557815,0.796323
10,0.362800,0.564387,0.799179


The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from you

TrainOutput(global_step=28010, training_loss=0.5184022260708112, metrics={'train_runtime': 2710.2002, 'train_samples_per_second': 165.346, 'train_steps_per_second': 10.335, 'total_flos': 2.978187750070272e+16, 'train_loss': 0.5184022260708112, 'epoch': 10.0})

In [ ]:
# Évaluation finale v4
from sklearn.metrics import classification_report
import numpy as np

predictions = trainer_v4.predict(test_ds3_torch)
preds  = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

print("=== RÉSULTATS FINAUX V4 ===\n")
print(classification_report(
    labels, preds,
    target_names=['negative', 'neutral', 'positive']
))

=== RÉSULTATS FINAUX V4 ===

              precision    recall  f1-score   support

    negative       0.87      0.90      0.88      1867
     neutral       0.74      0.83      0.79      1868
    positive       0.81      0.68      0.74      1867

    accuracy                           0.80      5602
   macro avg       0.81      0.80      0.80      5602
weighted avg       0.81      0.80      0.80      5602



In [ ]:
# Test généralisation — mêmes phrases nouvelles qu'avant
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_v4.to(device)
model_v4.eval()

new_test_phrases = [
    ("هاد الجاكيطة جاتني مقطعة من أول يوم، ماشي طبيعي", "negative"),
    ("التيليفون سخن بزاف وطفا وحدو، خايب بزاف", "negative"),
    ("3 أسابيع وما وصلنيش ولا خبر، خدمة سيئة", "negative"),
    ("الحجم مختلف على اللي كتبو، رجعتو ومازال ما ردوش فلوسي", "negative"),
    ("هاد الشامبو حرق شعري، ما ننصحش بيه أبدا", "negative"),
    ("علبة وصلت مهرسة وداخلها كسور، واضح ما حافظوش عليها", "negative"),
    ("الكاميرا ديالو كتصور بحال الضباب، خايبة بزاف", "negative"),
    ("هاد الطاجين طيب بزاف، عجبني من أول مرة", "positive"),
    ("شريت هاد الكريم وبشرتي تحسنت من أول أسبوع، ننصح بيه", "positive"),
    ("التوصيل جا من غدة لغدة والتغليف كان مزيان بزاف", "positive"),
    ("هاد الحذاء مريح بزاف حتى فالمشي الطويل", "positive"),
    ("الثمن مناسب بزاف مقارنة بالجودة، شريت جوج", "positive"),
    ("خدمة العملاء ردو عليا بسرعة وحلو المشكل فساعة", "positive"),
    ("هاد اللابتوب سريع بزاف وباطريتو تدوم نهار كامل", "positive"),
    ("المنتوج وصل فالوقت، لا مشكل لا ميزة خاصة", "neutral"),
    ("كيخدم مزيان لكن ما فيه حاجة تميزه", "neutral"),
    ("عادي كيما غيره، يصلح للاستخدام اليومي", "neutral"),
]

correct = 0
total   = len(new_test_phrases)

print("=== TEST GÉNÉRALISATION V4 ===\n")
for phrase, true_label in new_test_phrases:
    inputs = tokenizer(phrase, return_tensors="pt", truncation=True, max_length=128, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model_v4(**inputs)
        probs   = torch.softmax(outputs.logits, dim=-1)[0].cpu().numpy()

    pred_id    = int(np.argmax(probs))
    pred_label = id2label[pred_id]
    confidence = float(probs[pred_id]) * 100
    is_correct = pred_label == true_label
    if is_correct:
        correct += 1

    status = "✅" if is_correct else "❌"
    EMOJI  = {'positive': '😊', 'negative': '😠', 'neutral': '😐'}
    print(f"{status} {EMOJI[pred_label]} {pred_label:8s} ({confidence:5.1f}%) | attendu: {true_label:8s} | {phrase[:55]}")

print(f"\n{'='*60}")
print(f"Score généralisation : {correct}/{total} = {correct/total*100:.1f}%")
print(f"{'='*60}")

=== TEST GÉNÉRALISATION V4 ===

✅ 😠 negative ( 99.4%) | attendu: negative | هاد الجاكيطة جاتني مقطعة من أول يوم، ماشي طبيعي
✅ 😠 negative ( 99.7%) | attendu: negative | التيليفون سخن بزاف وطفا وحدو، خايب بزاف
✅ 😠 negative ( 99.6%) | attendu: negative | 3 أسابيع وما وصلنيش ولا خبر، خدمة سيئة
✅ 😠 negative ( 84.5%) | attendu: negative | الحجم مختلف على اللي كتبو، رجعتو ومازال ما ردوش فلوسي
✅ 😠 negative ( 99.5%) | attendu: negative | هاد الشامبو حرق شعري، ما ننصحش بيه أبدا
✅ 😠 negative ( 98.7%) | attendu: negative | علبة وصلت مهرسة وداخلها كسور، واضح ما حافظوش عليها
✅ 😠 negative ( 99.6%) | attendu: negative | الكاميرا ديالو كتصور بحال الضباب، خايبة بزاف
✅ 😊 positive ( 99.4%) | attendu: positive | هاد الطاجين طيب بزاف، عجبني من أول مرة
✅ 😊 positive ( 98.1%) | attendu: positive | شريت هاد الكريم وبشرتي تحسنت من أول أسبوع، ننصح بيه
✅ 😊 positive ( 98.1%) | attendu: positive | التوصيل جا من غدة لغدة والتغليف كان مزيان بزاف
✅ 😊 positive ( 98.9%) | attendu: positive | هاد الحذاء مريح بزاف حتى فالم

In [ ]:
# Sauvegarde v4
SAVE_PATH_V4 = "/content/drive/MyDrive/darija_sentiment_v4"
trainer_v4.save_model(SAVE_PATH_V4)
tokenizer.save_pretrained(SAVE_PATH_V4)
print(f"Modèle v4 sauvegardé dans {SAVE_PATH_V4}")

The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.


Modèle v4 sauvegardé dans /content/drive/MyDrive/darija_sentiment_v4


In [ ]:
import gradio as gr
import torch
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_v4.to(device)
model_v4.eval()

id2label = {0: 'negative', 1: 'neutral', 2: 'positive'}

def predict_sentiment(text):
    if not text.strip():
        return "أدخل نصاً", {}

    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model_v4(**inputs)
        probs   = torch.softmax(outputs.logits, dim=-1)[0].cpu().numpy()

    pred_id    = int(np.argmax(probs))
    pred_label = id2label[pred_id]
    confidence = float(probs[pred_id]) * 100

    scores = {
        "😠 Négatif" : float(probs[0]),
        "😐 Neutre"  : float(probs[1]),
        "😊 Positif" : float(probs[2]),
    }

    EMOJI  = {'positive': '😊 Positif', 'negative': '😠 Négatif', 'neutral': '😐 Neutre'}
    result = f"{EMOJI[pred_label]} — {confidence:.1f}%"
    return result, scores

with gr.Blocks(title="Darija Sentiment Analysis", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🇲🇦 تحليل المشاعر بالدارجة المغربية")
    gr.Markdown("**DarijaBERT + LoRA Fine-tuned** — Analyse de sentiment en Darija marocain")

    with gr.Row():
        with gr.Column():
            text_input = gr.Textbox(
                label="أدخل النص بالدارجة",
                placeholder="كتب هنا النص ديالك بالدارجة...",
                lines=4,
                rtl=True
            )
            submit_btn = gr.Button("تحليل المشاعر 🔍", variant="primary")

        with gr.Column():
            result_output = gr.Textbox(label="النتيجة")
            scores_output = gr.Label(label="نسب الثقة", num_top_classes=3)

    gr.Examples(
        examples=[
            ["هاد الخدمة خايبة بزاف ما عجبتنيش"],
            ["المنتوج مزيان بزاف وجا بسرعة"],
            ["عادي ماشي مزيان ماشي خايب"],
            ["التيليفون سخن بزاف وطفا وحدو خايب"],
            ["ننصح بيه بزاف جودة عالية وثمن مناسب"],
            ["خدمة العملاء ردو عليا بسرعة وحلو المشكل"],
        ],
        inputs=text_input
    )

    submit_btn.click(
        fn=predict_sentiment,
        inputs=text_input,
        outputs=[result_output, scores_output]
    )

demo.launch(share=True)

/tmp/ipykernel_3405/2150631799.py:36: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Darija Sentiment Analysis", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://903a387eeed0b8905e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Cibler les 2 problèmes identifiés :
# 1. Négation positive (ما كيعياش / ما كيتعطلش)
# 2. Phrases mixtes positif + تحفظ (مزيان لكن...)

extra_targeted = [
    # POSITIF — négation positive (ما + verbe = bon signe)
    {"cleaned_text": "هاد البلندر ما كيعياش أبدا، كيخدم بلا توقف", "label": "positive"},
    {"cleaned_text": "الباطري ما كتفرغش بسرعة، تدوم نهار كامل", "label": "positive"},
    {"cleaned_text": "ما سخنش ما وقفش، مودال قوي بزاف", "label": "positive"},
    {"cleaned_text": "ما كيتعطلش أبدا حتى مع الاستخدام الكثير", "label": "positive"},
    {"cleaned_text": "ما عندوش أي مشكل من شهرين، ننصح بيه", "label": "positive"},
    {"cleaned_text": "ما دخل عليا أي بوغا منذ شريتو، ممتاز", "label": "positive"},
    {"cleaned_text": "ما وقفش ولا مرة حتى مع الكهرباء الضعيفة", "label": "positive"},
    {"cleaned_text": "ما كيشخنش ما كيتعطلش، رائع بصح", "label": "positive"},
    {"cleaned_text": "ما سخنتش يدي ولو مرة، تصميم ممتاز للسلامة", "label": "positive"},
    {"cleaned_text": "ما توقفش ولا مرة فالعمل، موثوق بزاف", "label": "positive"},
    {"cleaned_text": "ما فيه ضجيج ما فيه اهتزاز، هادو علامات الجودة", "label": "positive"},
    {"cleaned_text": "ما كيدخلش الماء ما كيتخدشش، متين بزاف", "label": "positive"},
    {"cleaned_text": "ما كتعبتش فالتركيب أبدا، كل شي واضح", "label": "positive"},
    {"cleaned_text": "ما تأخرش التوصيل ولا يوم واحد، وصل بسرعة", "label": "positive"},
    {"cleaned_text": "ما كيعياش حتى مع الاستخدام المتواصل", "label": "positive"},

    # NEUTRE — positif + تحفظ (لكن / غير أن / إلا أن)
    {"cleaned_text": "وصل فالوقت والتغليف سليم لكن المنتوج عادي", "label": "neutral"},
    {"cleaned_text": "الجودة مقبولة لكن ما فيه حاجة تبهرك", "label": "neutral"},
    {"cleaned_text": "كيخدم مزيان لكن الثمن غالي شوية عليه", "label": "neutral"},
    {"cleaned_text": "التوصيل سريع لكن التغليف كان ضعيف", "label": "neutral"},
    {"cleaned_text": "المنتوج سليم لكن ما تجاوزش توقعاتي", "label": "neutral"},
    {"cleaned_text": "مزيان لكن ما فيه حاجة تميزه عن المنافسين", "label": "neutral"},
    {"cleaned_text": "الجودة كويسة لكن للثمن ديالو كنت نتوقع أحسن", "label": "neutral"},
    {"cleaned_text": "كيخدم بلا مشاكل لكن بدون أي ميزة خاصة", "label": "neutral"},
    {"cleaned_text": "وصل سليم لكن الألوان مختلفة شوية على الصورة", "label": "neutral"},
    {"cleaned_text": "سريع فالتوصيل لكن الجودة في المستوى المتوسط", "label": "neutral"},
    {"cleaned_text": "التصميم زوين لكن الأداء عادي", "label": "neutral"},
    {"cleaned_text": "مناسب للثمن لكن مانيش غادي نشريه مرة أخرى", "label": "neutral"},
    {"cleaned_text": "يخدم الخدمة ديالو لكن ما أبهرنيش", "label": "neutral"},
    {"cleaned_text": "عجبني الشكل لكن الجودة الداخلية متوسطة", "label": "neutral"},
    {"cleaned_text": "مقبول لكن توقعت أحسن من هاد الماركة", "label": "neutral"},
    {"cleaned_text": "التغليف ممتاز لكن المنتوج نفسو عادي بحاله", "label": "neutral"},
    {"cleaned_text": "خدمة العملاء كويسة لكن المنتوج متوسط", "label": "neutral"},
    {"cleaned_text": "الحجم صح لكن الخامة ماشي كيما توقعت", "label": "neutral"},
    {"cleaned_text": "يصلح للاستخدام لكن مانيش متحمس بزاف عليه", "label": "neutral"},
    {"cleaned_text": "وصل بسرعة لكن ما فيه حاجة تستاهل التوصية بيه", "label": "neutral"},
]

print(f"Exemples ciblés ajoutés : {len(extra_targeted)}")
print(f"  - Positif (négation positive) : {sum(1 for x in extra_targeted if x['label'] == 'positive')}")
print(f"  - Neutre (positif + لكن)      : {sum(1 for x in extra_targeted if x['label'] == 'neutral')}")

Exemples ciblés ajoutés : 35
  - Positif (négation positive) : 15
  - Neutre (positif + لكن)      : 20


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from datasets import Dataset
from torch.utils.data import Dataset as TorchDataset
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from peft import LoraConfig, get_peft_model, TaskType
import evaluate, numpy as np, torch

# Dataset original
df_original = pd.read_csv("./NLP Dataset/sentences/darija_clean.csv")
df_original['label'] = df_original['label'].str.strip().str.lower()
label_map = {
    'positive': 'positive', 'positif': 'positive',
    'negative': 'negative', 'négatif': 'negative',
    'neutral':  'neutral',  'neutre':  'neutral',
}
df_original = df_original[df_original['label'].isin(label_map)][['cleaned_text','label']].copy()
df_original['label'] = df_original['label'].map(label_map)

# Fusion de toutes les sources
df_jumia_clean      = df_jumia[['cleaned_text','label']].copy()
df_neutral_extra_df = pd.DataFrame(neutral_extra)
df_targeted         = pd.DataFrame(extra_targeted)

df_all = pd.concat([
    df_original,
    df_jumia_clean,
    df_neutral_extra_df,
    df_targeted
], ignore_index=True)

# Rééquilibrage
df_pos = df_all[df_all['label'] == 'positive']
df_neg = df_all[df_all['label'] == 'negative']
df_neu = df_all[df_all['label'] == 'neutral']
target = len(df_pos)

df_neg_up = resample(df_neg, replace=True, n_samples=target, random_state=42)
df_neu_up = resample(df_neu, replace=True, n_samples=target, random_state=42)

df_balanced = pd.concat([df_pos, df_neg_up, df_neu_up], ignore_index=True).sample(frac=1, random_state=42)

label2id = {'negative': 0, 'neutral': 1, 'positive': 2}
id2label  = {v: k for k, v in label2id.items()}
df_balanced['label_id'] = df_balanced['label'].map(label2id)

print("Distribution finale :")
print(df_balanced['label'].value_counts())
print(f"Total : {len(df_balanced)}")

# Splits
train_df4, temp_df4 = train_test_split(df_balanced, test_size=0.2, random_state=42, stratify=df_balanced['label'])
val_df4,  test_df4  = train_test_split(temp_df4, test_size=0.5, random_state=42, stratify=temp_df4['label'])

# Tokenisation
def tokenize(batch):
    return tokenizer(batch["cleaned_text"], truncation=True, padding="max_length", max_length=128)

train_ds4 = Dataset.from_pandas(train_df4[['cleaned_text','label_id']].rename(columns={'label_id':'labels'})).map(tokenize, batched=True)
val_ds4   = Dataset.from_pandas(val_df4[['cleaned_text','label_id']].rename(columns={'label_id':'labels'})).map(tokenize, batched=True)
test_ds4  = Dataset.from_pandas(test_df4[['cleaned_text','label_id']].rename(columns={'label_id':'labels'})).map(tokenize, batched=True)

class DarijaDataset(TorchDataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "input_ids"      : torch.tensor(item["input_ids"],      dtype=torch.long),
            "attention_mask" : torch.tensor(item["attention_mask"], dtype=torch.long),
            "labels"         : torch.tensor(item["labels"],         dtype=torch.long),
        }

train_ds4_torch = DarijaDataset(train_ds4)
val_ds4_torch   = DarijaDataset(val_ds4)
test_ds4_torch  = DarijaDataset(test_ds4)

print(f"\nTrain : {len(train_ds4_torch)} | Val : {len(val_ds4_torch)} | Test : {len(test_ds4_torch)}")

# Nouveau modèle propre
model_v5 = AutoModelForSequenceClassification.from_pretrained(
    "SI2M-Lab/DarijaBERT",
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
    torch_dtype=torch.float32
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query", "value", "key"],
    bias="none"
)

model_v5 = get_peft_model(model_v5, lora_config)
model_v5.print_trainable_parameters()

# Entraînement
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=-1)
    return accuracy.compute(predictions=preds, references=eval_pred.label_ids)

training_args_v5 = TrainingArguments(
    output_dir                  = "./model_output_v5",
    num_train_epochs            = 10,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    learning_rate               = 2e-4,
    weight_decay                = 0.01,
    warmup_steps                = 500,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "accuracy",
    greater_is_better           = True,
    fp16                        = torch.cuda.is_available(),
    logging_steps               = 100,
    report_to                   = "none",
    seed                        = 42,
)

trainer_v5 = Trainer(
    model           = model_v5,
    args            = training_args_v5,
    train_dataset   = train_ds4_torch,
    eval_dataset    = val_ds4_torch,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)],
)

print(f"\nFine-tuning v5 — {len(train_ds4_torch)} exemples, max 10 epochs")
print(f"GPU : {'✅ ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ CPU'}")
trainer_v5.train()

Distribution finale :
label
positive    18687
negative    18687
neutral     18687
Name: count, dtype: int64
Total : 56061


Map:   0%|          | 0/44848 [00:00<?, ? examples/s]

Map:   0%|          | 0/5606 [00:00<?, ? examples/s]

Map:   0%|          | 0/5607 [00:00<?, ? examples/s]

The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.


...

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at SI2M-Lab/DarijaBERT and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 887,043 || all params: 148,370,694 || trainable%: 0.5979

Fine-tuning v5 — 44848 exemples, max 10 epochs
GPU : ✅ Tesla T4


Epoch,Training Loss,Validation Loss,Accuracy
1,0.708200,0.671810,0.708527
2,0.644900,0.629108,0.729575
3,0.583100,0.593490,0.755797
4,0.511700,0.590705,0.766143
5,0.485800,0.560629,0.782911
6,0.448100,0.560248,0.788619
7,0.449500,0.550284,0.796825
8,0.393500,0.556741,0.800036
9,0.365000,0.574768,0.800571
10,0.369400,0.572488,0.800392


The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.
The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from you

TrainOutput(global_step=28030, training_loss=0.51054476354533, metrics={'train_runtime': 2730.3427, 'train_samples_per_second': 164.258, 'train_steps_per_second': 10.266, 'total_flos': 2.980580295794688e+16, 'train_loss': 0.51054476354533, 'epoch': 10.0})

In [ ]:
# Évaluation finale v5
from sklearn.metrics import classification_report
import numpy as np
import torch

model_v5.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_v5.to(device)

all_preds, all_labels = [], []

from torch.utils.data import DataLoader
test_loader = DataLoader(test_ds4_torch, batch_size=32)

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        outputs = model_v5(input_ids=input_ids, attention_mask=attention_mask)
        preds   = torch.argmax(outputs.logits, dim=-1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("=== RÉSULTATS FINAUX V5 ===")
print(classification_report(
    all_labels, all_preds,
    target_names=["negative", "neutral", "positive"]
))

# Test généralisation
print("\n=== TEST GÉNÉRALISATION V5 ===")
test_phrases = [
    ("هاد البلندر ما كيعياش أبدا، سريع ونظيف",                    "positive"),
    ("وصل فالوقت والتغليف سليم، لكن المنتوج ما فيه حاجة تبهرك",   "neutral"),
    ("الطونيبور ديالي وصل مكسور وما خدمش حتى مرة",                "negative"),
    ("غير شريت هاد البلندر وطيبت بيه أول مرة، ننصح بيه",          "positive"),
    ("دفعت تمن غالي وجاني شي حاجة بحال السوق الشعبي",             "negative"),
    ("شريت هاد الكرسي ليولدي وعجبو بزاف، جودته مزيانة",           "positive"),
    ("كيخدم مزيان لكن الثمن غالي شوية عليه",                      "neutral"),
    ("ليلة كاملة نحاول نرجع المنتوج وخدمة العملاء ما كتردش",      "negative"),
    ("التصميم زوين لكن الأداء عادي",                               "neutral"),
    ("الباطري ما كتفرغش بسرعة، تدوم نهار كامل",                   "positive"),
]

EMOJI = {'positive': '😊 positive', 'negative': '😠 negative', 'neutral': '😐 neutral'}
correct = 0

for text, expected in test_phrases:
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                       max_length=128, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model_v5(**inputs)
        probs   = torch.softmax(outputs.logits, dim=-1)[0].cpu().numpy()

    pred_id    = int(np.argmax(probs))
    pred_label = id2label[pred_id]
    confidence = float(probs[pred_id]) * 100
    ok         = "✅" if pred_label == expected else "❌"
    if pred_label == expected:
        correct += 1

    print(f"{ok} {EMOJI[pred_label]} ({confidence:5.1f}%) | attendu: {expected:8s} | {text[:60]}")

print(f"\n{'='*60}")
print(f"Score généralisation : {correct}/{len(test_phrases)} = {correct/len(test_phrases)*100:.1f}%")
print(f"{'='*60}")

=== RÉSULTATS FINAUX V5 ===
              precision    recall  f1-score   support

    negative       0.90      0.85      0.87      1869
     neutral       0.73      0.86      0.79      1869
    positive       0.79      0.69      0.74      1869

    accuracy                           0.80      5607
   macro avg       0.81      0.80      0.80      5607
weighted avg       0.81      0.80      0.80      5607


=== TEST GÉNÉRALISATION V5 ===
✅ 😊 positive ( 90.3%) | attendu: positive | هاد البلندر ما كيعياش أبدا، سريع ونظيف
✅ 😐 neutral ( 89.7%) | attendu: neutral  | وصل فالوقت والتغليف سليم، لكن المنتوج ما فيه حاجة تبهرك
✅ 😠 negative ( 99.0%) | attendu: negative | الطونيبور ديالي وصل مكسور وما خدمش حتى مرة
✅ 😊 positive ( 71.0%) | attendu: positive | غير شريت هاد البلندر وطيبت بيه أول مرة، ننصح بيه
✅ 😠 negative ( 59.2%) | attendu: negative | دفعت تمن غالي وجاني شي حاجة بحال السوق الشعبي
✅ 😊 positive ( 99.6%) | attendu: positive | شريت هاد الكرسي ليولدي وعجبو بزاف، جودته مزيانة
✅ 😐 neutral ( 9

In [ ]:
# Test phrases totalement nouvelles — domaines jamais vus
test_new = [
    # Domaine : École / Université
    ("الكتاب ديال الدارس وصل مبلل بالماء وماقدرتش نقراه",           "negative"),
    ("الأستاذ شرح بزاف مزيان وفهمت كلشي من أول مرة",               "positive"),
    ("الدرس كان عادي، لا مميز لا رديء",                              "neutral"),

    # Domaine : Transport / Livraison
    ("السيارة ديال التوصيل جات متأخرة ساعتين بلا إشعار",            "negative"),
    ("الطاكسي وصل قبل الوقت المحدد وكان نظيف بزاف",                 "positive"),

    # Domaine : Santé / Pharmacie
    ("الدواء ما خدمش معايا بلا ما نحس بأي تحسن",                    "negative"),
    ("الكريم الجديد خف على البشرة وما دارش حساسية",                  "positive"),
    ("الشامبو عادي كيفاش نقول، ما يفرقش على غيره",                   "neutral"),

    # Domaine : Alimentation / Cuisine
    ("الطاجين وصل بارد وبدون الصلصة اللي كتبو عليها",               "negative"),
    ("العسل ديالهم طبيعي بصح وريحتو زوينة بزاف",                    "positive"),
]

print("=== TEST DOMAINES NOUVEAUX ===\n")
correct = 0

for text, expected in test_new:
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                       max_length=128, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model_v5(**inputs)
        probs   = torch.softmax(outputs.logits, dim=-1)[0].cpu().numpy()

    pred_label = id2label[int(np.argmax(probs))]
    confidence = float(np.max(probs)) * 100
    ok = "✅" if pred_label == expected else "❌"
    if pred_label == expected:
        correct += 1

    print(f"{ok} 【{pred_label:8s}】({confidence:5.1f}%) | attendu: {expected:8s} | {text}")

print(f"\n{'='*60}")
print(f"Score domaines nouveaux : {correct}/{len(test_new)} = {correct/len(test_new)*100:.1f}%")
print(f"{'='*60}")

=== TEST DOMAINES NOUVEAUX ===

✅ 【negative】( 98.3%) | attendu: negative | الكتاب ديال الدارس وصل مبلل بالماء وماقدرتش نقراه
✅ 【positive】( 93.1%) | attendu: positive | الأستاذ شرح بزاف مزيان وفهمت كلشي من أول مرة
✅ 【neutral 】( 98.4%) | attendu: neutral  | الدرس كان عادي، لا مميز لا رديء
❌ 【positive】( 61.4%) | attendu: negative | السيارة ديال التوصيل جات متأخرة ساعتين بلا إشعار
✅ 【positive】( 92.0%) | attendu: positive | الطاكسي وصل قبل الوقت المحدد وكان نظيف بزاف
✅ 【negative】( 47.6%) | attendu: negative | الدواء ما خدمش معايا بلا ما نحس بأي تحسن
❌ 【neutral 】( 77.8%) | attendu: positive | الكريم الجديد خف على البشرة وما دارش حساسية
❌ 【positive】( 53.2%) | attendu: neutral  | الشامبو عادي كيفاش نقول، ما يفرقش على غيره
❌ 【positive】( 96.4%) | attendu: negative | الطاجين وصل بارد وبدون الصلصة اللي كتبو عليها
✅ 【positive】( 97.7%) | attendu: positive | العسل ديالهم طبيعي بصح وريحتو زوينة بزاف

Score domaines nouveaux : 6/10 = 60.0%


In [ ]:
# Sauvegarde modèle v5 final
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

SAVE_PATH = "/content/drive/MyDrive/darija_sentiment/model_v5_final"
import os; os.makedirs(SAVE_PATH, exist_ok=True)

model_v5.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Modèle sauvegardé : {SAVE_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


The `xla_device` argument has been deprecated in v4.4.0 of Transformers. It is ignored and you can safely remove it from your `config.json` file.


Modèle sauvegardé : /content/drive/MyDrive/darija_sentiment/model_v5_final


In [1]:
import gradio as gr
import torch
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_v5.to(device)
model_v5.eval()

id2label = {0: 'negative', 1: 'neutral', 2: 'positive'}

def predict_sentiment(text):
    if not text or not text.strip():
        return "أدخل نصاً", {}

    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model_v5(**inputs)
        probs   = torch.softmax(outputs.logits, dim=-1)[0].cpu().numpy()

    pred_id    = int(np.argmax(probs))
    pred_label = id2label[pred_id]
    confidence = float(probs[pred_id]) * 100

    EMOJI  = {'positive': '😊 Positif', 'negative': '😡 Négatif', 'neutral': '😐 Neutre'}
    result = f"{EMOJI[pred_label]} — {confidence:.1f}%"

    scores = {
        "😡 Négatif" : float(probs[0]),
        "😊 Positif" : float(probs[2]),
        "😐 Neutre"  : float(probs[1]),
    }
    return result, scores

css = """
#title { text-align: right; direction: rtl; font-size: 28px; font-weight: bold; }
#subtitle { text-align: right; direction: rtl; }
#input-box textarea { direction: rtl; text-align: right; font-size: 16px; }
#result-box textarea { direction: rtl; text-align: right; font-size: 18px; font-weight: bold; }
#submit-btn { background: #5865f2 !important; color: white !important; font-size: 18px !important; border-radius: 10px !important; }
"""

with gr.Blocks(css=css, title="Darija Sentiment Analysis") as demo:

    gr.HTML('<div id="title">تحليل المشاعر بالدارجة المغربية 🇲🇦</div>')
    gr.HTML('<div id="subtitle"><b>DarijaBERT + LoRA Fine-tuned</b> — Analyse de sentiment en Darija marocain</div>')
    gr.HTML("<br>")

    with gr.Row():
        with gr.Column(scale=1):
            text_input = gr.Textbox(
                label="أدخل النص بالدارجة",
                placeholder="أدخل النص بالدارجة...",
                lines=5,
                rtl=True,
                elem_id="input-box"
            )
            submit_btn = gr.Button(
                "🔍 تحليل المشاعر",
                variant="primary",
                elem_id="submit-btn"
            )

        with gr.Column(scale=1):
            result_output = gr.Textbox(
                label="النتيجة",
                elem_id="result-box",
                interactive=False
            )
            scores_output = gr.Label(
                label="نسب الثقة",
                num_top_classes=3
            )

    gr.Examples(
        examples=[
            ["هاد الخدمة خايبة بزاف ما عجبتنيش"],
            ["المنتوج مزيان بزاف وجا بسرعة"],
            ["عادي ماشي مزيان ماشي خايب"],
            ["التيليفون سخن بزاف وطفا وحدو خايب"],
            ["ننصح بيه بزاف جودة عالية وثمن مناسب"],
            ["خدمة العملاء ردو عليا بسرعة وحلو المشكل"],
        ],
        inputs=text_input,
        label="Examples"
    )

    submit_btn.click(
        fn=predict_sentiment,
        inputs=text_input,
        outputs=[result_output, scores_output]
    )

demo.launch(share=True)

NameError: name 'model_v5' is not defined